# How to use R-XIMO for explainable interactive multiobjective optimization

This guide walks through using **R-XIMO** to solve the river pollution
problem interactively, with SHAP-based explanations guiding each
preference adjustment. R-XIMO produces, for every iteration, a textual
explanation of why the current solution looks the way it does and a
concrete suggestion of which objective to relax in order to improve a
target objective.

We follow the illustrative flow from Misitano _et al._ (2022): start
from a very demanding reference point (the ideal), then iteratively
adjust the reference point in the direction R-XIMO recommends.

See also the explanation page on
[explainable multiobjective optimization](../explanation/explainable_moo.md)
for the conceptual background.


In [ ]:
import numpy as np
import polars as pl
from scipy.spatial import cKDTree

from desdeo.explanations import RXIMOResult, ShapExplainer, run_rximo
from desdeo.problem.testproblems import river_pollution_problem_discrete

## Load the river pollution problem

The 5-objective discrete variant has four maximized objectives
(`f1`–`f4`: dissolved oxygen and ROIs) and one minimized objective
(`f5`: BOD deviation). All SHAP analysis below is done in **minimization
form**, so we flip the maximized columns by multiplying with $-1$.


In [ ]:
problem = river_pollution_problem_discrete(five_objective_variant=True)

OUTPUT_SYMBOLS = ["f1", "f2", "f3", "f4", "f5"]
INPUT_SYMBOLS = ["z1", "z2", "z3", "z4", "z5"]
objective_names = [obj.name for obj in problem.objectives]
sign = np.array([1.0 if obj.maximize else -1.0 for obj in problem.objectives])

for name, mx in zip(objective_names, sign > 0):
    print(f"  {name:20s} ({'maximize' if mx else 'minimize'})")

## Prepare the Pareto front data

We pull the discrete representation, flip the maximized columns, and
compute the ideal and nadir points in minimization form. These bounds
define the box from which we will draw reference points later.


In [ ]:
rep = problem.discrete_representation
pf_orig = np.column_stack([np.asarray(rep.objective_values[s], dtype=float) for s in OUTPUT_SYMBOLS])
pf_min = pf_orig * (-sign)

ideal_min = pf_min.min(axis=0)
nadir_min = pf_min.max(axis=0)

print(f"Pareto front size: {pf_min.shape[0]}")
print(f"Ideal (min form): {np.round(ideal_min, 3)}")
print(f"Nadir (min form): {np.round(nadir_min, 3)}")

## Build the black-box and the SHAP explainer

Our "black-box" is the function that maps a reference point to a
Pareto optimal solution. For the discrete problem we approximate it by
a nearest-neighbor lookup on the front. We then sample 200 random
reference points uniformly between the ideal and nadir points, pair
them with their solutions, and feed both to a `ShapExplainer`.

The explainer's internal `evaluate` is replaced with the true black-box
so SHAP queries hit the actual model. The background dataset is a small
subset of the Pareto front, with only five features `shap.Explainer`
automatically uses the Exact algorithm, so background size mostly
affects the base value, not SHAP value accuracy.


In [ ]:
pf_tree = cKDTree(pf_min)


def black_box(ref_points: np.ndarray) -> np.ndarray:
    arr = np.asarray(ref_points, dtype=float)
    single = arr.ndim == 1
    if single:
        arr = arr.reshape(1, -1)
    _, idx = pf_tree.query(arr)
    out = pf_min[idx]
    return out[0] if single else out


RNG_SEED = 67
N_TRAIN = 200
N_BACKGROUND = 25
rng = np.random.default_rng(RNG_SEED)

train_refs = rng.uniform(low=ideal_min, high=nadir_min, size=(N_TRAIN, 5))
train_sols = black_box(train_refs)

training_columns: dict[str, np.ndarray] = {sym: train_refs[:, i] for i, sym in enumerate(INPUT_SYMBOLS)}
for i, sym in enumerate(OUTPUT_SYMBOLS):
    training_columns[sym] = train_sols[:, i]
training_df = pl.DataFrame(training_columns)

explainer = ShapExplainer(
    problem_data=training_df,
    input_symbols=INPUT_SYMBOLS,
    output_symbols=OUTPUT_SYMBOLS,
)
explainer.evaluate = black_box  # use the real black-box for SHAP queries

bg_idx = rng.choice(pf_min.shape[0], size=N_BACKGROUND, replace=False)
background_df = pl.DataFrame({sym: pf_min[bg_idx, i] for i, sym in enumerate(INPUT_SYMBOLS)})
explainer.setup(background_data=background_df)

print(f"Trained on {N_TRAIN} reference-point/solution pairs.")
print(f"Background: {N_BACKGROUND} sampled Pareto points.")

We also define a couple of helper functions for printing reference
points and solutions in their original orientation, and for running an
R-XIMO iteration end-to-end.


In [ ]:
def to_orig(min_vector: np.ndarray) -> np.ndarray:
    """Flip a (k,) min-form vector back to its original orientation."""
    return min_vector * (-sign)


def show_iteration(ref_min: np.ndarray, sol_min: np.ndarray, label: str) -> None:
    ref_orig = to_orig(ref_min)
    sol_orig = to_orig(sol_min)
    width = max(len(n) for n in objective_names)
    print(f"\n{label}")
    print(f"  {'Objective':<{width}}  {'Reference':>10}  {'Solution':>10}")
    for i, name in enumerate(objective_names):
        print(f"  {name:<{width}}  {ref_orig[i]:>10.3f}  {sol_orig[i]:>10.3f}")


iterations: list[dict] = []


def run_iteration(ref_min: np.ndarray, target_index: int, label: str) -> RXIMOResult:
    sol_min = black_box(ref_min)
    show_iteration(ref_min, sol_min, label=label)
    result = run_rximo(
        explainer=explainer,
        reference_point=ref_min,
        solution=sol_min,
        target_index=target_index,
        objective_names=objective_names,
    )
    print(f"\n  Target: {objective_names[target_index]}")
    print(f"  Rival : {objective_names[result.rival_index]}  (case {result.explanation_index})")
    print(f"  Explanation: {result.explanation}")
    print(f"  Suggestion : {result.suggestion}")
    iterations.append(
        {
            "label": label,
            "ref": ref_min.copy(),
            "sol": sol_min.copy(),
            "target": target_index,
            "rival": result.rival_index,
            "case": result.explanation_index,
        }
    )
    return result

## Iteration 1: start from a neutral reference point

We start from a neutral reference point: the midpoint between the ideal and the
nadir. The DM looks at the resulting solution and decides to push **DO city**
($f_1$) further. R-XIMO returns a rival objective and the corresponding textual
explanation.


In [ ]:
ref1 = (ideal_min + nadir_min) / 2.0
res1 = run_iteration(ref1, target_index=0, label="Iteration 1: neutral reference point")

## Iteration 2: follow the suggestion

We move the target component a little more demanding (toward the ideal)
and relax the rival component a little (toward the nadir). The DM
switches focus to **DO municipality** ($f_2$).


In [ ]:
DELTA_FRACTION = 0.10
delta_vec = DELTA_FRACTION * (nadir_min - ideal_min)


def follow_suggestion(prev_ref: np.ndarray, target: int, rival: int) -> np.ndarray:
    new_ref = prev_ref.copy()
    new_ref[target] = max(new_ref[target] - delta_vec[target], ideal_min[target])
    new_ref[rival] = min(new_ref[rival] + delta_vec[rival], nadir_min[rival])
    return new_ref


ref2 = follow_suggestion(ref1, res1.target_index, res1.rival_index)
res2 = run_iteration(ref2, target_index=1, label="Iteration 2: improve f1, impair rival")

## Iterations 3 and 4: keep iterating

We continue the same pattern. Each iteration: take the previous
reference point, follow R-XIMO's suggestion, pick a new target, and ask
for a new explanation.


In [ ]:
ref3 = follow_suggestion(ref2, res2.target_index, res2.rival_index)
res3 = run_iteration(ref3, target_index=2, label="Iteration 3: focus on ROI fishery")

In [ ]:
ref4 = follow_suggestion(ref3, res3.target_index, res3.rival_index)
res4 = run_iteration(ref4, target_index=4, label="Iteration 4: focus on BOD deviation")

## Summary table

All iterations side by side, with reference points and solutions in the
original objective orientation.


In [ ]:
header = (
    ["iteration", "target", "rival", "case"]
    + [f"ref:{n}" for n in objective_names]
    + [f"sol:{n}" for n in objective_names]
)
rows = []
for k, it in enumerate(iterations, start=1):
    ref_orig = to_orig(it["ref"])
    sol_orig = to_orig(it["sol"])
    rows.append(
        [
            k,
            objective_names[it["target"]],
            objective_names[it["rival"]],
            it["case"],
            *ref_orig.tolist(),
            *sol_orig.tolist(),
        ]
    )

summary_df = pl.DataFrame({h: [r[i] for r in rows] for i, h in enumerate(header)})
summary_df

## Switching the baseline

The base value $\phi_0$ that anchors the SHAP decomposition depends on
the background distribution. To rewrite the explanations _relative to a
specific point_, for example, the current solution, call
`setup_with_baseline()` with that point. The Exact SHAP algorithm runs
regardless of background size for problems with at most about ten
features, so the SHAP values themselves remain exact; only $\phi_0$
changes.


In [ ]:
current_ref = iterations[-1]["ref"]
explainer.setup_with_baseline(current_ref)
res_baseline = run_rximo(
    explainer=explainer,
    reference_point=current_ref,
    solution=iterations[-1]["sol"],
    target_index=iterations[-1]["target"],
    objective_names=objective_names,
)
print("With current reference point as baseline:")
print(f"  Rival      : {objective_names[res_baseline.rival_index]}  (case {res_baseline.explanation_index})")
print(f"  Suggestion : {res_baseline.suggestion}")

## References

- Lundberg, S. M., & Lee, S.-I. (2017). _A Unified Approach to Interpreting
  Model Predictions._ In _Advances in Neural Information Processing Systems
  30_ (pp. 4768–4777).
- Misitano, G., Afsar, B., Lárraga, G., & Miettinen, K. (2022). _Towards
  explainable interactive multiobjective optimization: R-XIMO._ Autonomous
  Agents and Multi-Agent Systems, 36(43).
